# CARA-lite integrated pipeline
Runs `scripts/16_run_full_cara_lite_experiment.py` end-to-end on Google Colab.


## 1. Mount Drive & clone repo
Set `USE_DRIVE=True` to persist `data/`, `results/`, `figures/` across Colab runtime resets. 
Set `REPO_URL` to override the auto-detected git remote.


In [ ]:
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/CARA-FinSent"  #@param {type:"string"}
REPO_URL = ""  #@param {type:"string"}  # leave blank to auto-detect from this notebook's repo

import os, subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    WORK_DIR = Path(DRIVE_DIR)
else:
    WORK_DIR = Path("/content")

REPO_DIR = WORK_DIR / "cara-finsent-experiments"
if not REPO_DIR.exists():
    if not REPO_URL:
        # Best-effort auto-detect: try the repo this notebook lives in (works when notebook is opened from GitHub).
        REPO_URL = os.environ.get("REPO_URL", "")
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL above (e.g. https://github.com/<user>/cara-finsent-experiments.git)")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working in:", os.getcwd())


## 2. Install dependencies


In [ ]:
!pip -q install -r requirements.txt


## 3a. Pick the input dataset


In [ ]:
## Resolve the latest standardized CSV (run notebook 00 first if missing)
from pathlib import Path
matches = sorted(Path("data/processed").glob("combined_standardized_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
assert matches, "No standardized CSV found in data/processed/. Run 00_prepare_phrasebank_fiqa.ipynb first."
DATA = str(matches[0])
print("DATA =", DATA)


## 3b. Configure parameters


In [ ]:
BASE_MODEL = "logreg"  #@param ["logreg","svm"]
TOP_K = 3  #@param {type:"integer"}
EXTERNAL_CORPUS = ""  #@param {type:"string"}
NO_RETRIEVAL = False  #@param {type:"boolean"}
NO_STRUCTURED = False  #@param {type:"boolean"}
NO_AGREEMENT = False  #@param {type:"boolean"}
NO_CALIBRATION = False  #@param {type:"boolean"}
MAX_ROWS = 0  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
argv = ["--data", DATA, "--base_model", BASE_MODEL, "--top_k", str(TOP_K), "--seed", str(SEED)]
if EXTERNAL_CORPUS: argv += ["--external_corpus_csv", EXTERNAL_CORPUS]
for flag, on in [("--no_retrieval", NO_RETRIEVAL), ("--no_structured", NO_STRUCTURED),
                  ("--no_agreement", NO_AGREEMENT), ("--no_calibration", NO_CALIBRATION)]:
    if on: argv.append(flag)
if MAX_ROWS: argv += ["--max_rows", str(MAX_ROWS)]


## 3c. Run `16_run_full_cara_lite_experiment.py`


In [ ]:
import sys, runpy
sys.argv = ['scripts/16_run_full_cara_lite_experiment.py'] + argv
print("Running:", " ".join(sys.argv))
runpy.run_path("scripts/16_run_full_cara_lite_experiment.py", run_name="__main__")


## 4. Inspect latest `cara_lite_summary_*.csv`


In [ ]:
import pandas as pd
from pathlib import Path
matches = sorted(Path("results").rglob("cara_lite_summary_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
assert matches, "No summary CSV found — did the script run?"
latest = matches[0]
print("Latest:", latest)
df = pd.read_csv(latest)
df


## ⤓ Download results
Zip `results/` + `figures/` for sharing.


In [ ]:
import shutil, datetime
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
archive = shutil.make_archive(f"cara_results_{ts}", "zip", root_dir=".", base_dir="results")
print("Created:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
